# 04.5 — The bottleneck

Between the encoder and the latent space sits the **bottleneck**: a single `Linear` layer that compresses the encoder's hidden representation down to the latent dimension. It's the simplest module in the architecture — and worth its own notebook precisely because its *simplicity is deliberate*, and its exact shape is dictated by the VAE that follows it ([Module 06](../README.md)).

This notebook covers:

* What the bottleneck does and why it's just one FC layer.
* The per-timestep application that preserves the sequence axis.
* The `2 × latent` output width and what the VAE does with it (a forward glance).
* `LinearBottleneck` vs `PassthroughBottleneck` — the two cases.

**Prerequisites:** [04.2 simple encoder](04.2_building_a_simple_encoder.ipynb), [02.6 nn.Module](../02_numpy_and_pytorch_basics/02.6_nn_module_vs_layergraph.ipynb).

## Section 1 — What MATLAB does

`cgg_selectBottleNeck` always appends a final fully-connected layer with He initialization after the encoder's block stack:

```matlab
layers = [layers; fullyConnectedLayer(HiddenSizeBottleNeck, ...
    "WeightsInitializer", "he")];
```

`HiddenSizeBottleNeck` is the last entry of `HiddenSizes` — the encoder narrows down to it, and this final FC produces the latent representation the VAE samples from. For the stochastic encoder, that width is `2 × latent_dim` (mean and log-variance stacked). The `"he"` initializer is deliberate — [04.8](04.8_weight_initialization_he_vs_pytorch_defaults.ipynb) covers why.

## Section 2 — The Python concepts you need

### 2.1 — It's one Linear layer

In [ ]:
import torch
from neural_data_decoding.models.bottleneck import LinearBottleneck

bottleneck = LinearBottleneck(in_features=16, hidden_size=8)
print(bottleneck)
x = torch.randn(2, 5, 16)          # (batch, windows, encoder_hidden=16)
print("in :", tuple(x.shape))
print("out:", tuple(bottleneck(x).shape), "(batch, windows, bottleneck=8)")

That's the whole module — a `Linear(16 → 8)`. No nonlinearity, no dropout: the bottleneck's job is a clean linear projection into the latent space, and the VAE's sampling layer supplies the stochasticity. Keeping it linear is a design choice, not an oversight.

### 2.2 — Per-timestep, preserving the sequence

Notice the output kept the `windows` axis (`5`). A `Linear` layer in PyTorch applies to the **last axis only**, broadcasting over all leading axes — so a `(batch, windows, features)` input gets the projection applied independently at each window, and the sequence structure survives. This matters because the Deep LSTM classifier downstream still wants a *sequence*, not a single vector:

In [ ]:
# The Linear touches ONLY the last axis; batch and windows pass through untouched:
for shape in [(16,), (2, 16), (2, 5, 16), (2, 5, 3, 16)]:
    out = bottleneck(torch.randn(*shape))
    print(f"  input {str(shape):16} → output {tuple(out.shape)}  (only last axis 16→8)")

### 2.3 — The `2 × latent` width (a forward glance at the VAE)

For the stochastic (variational) encoder, the bottleneck outputs **twice** the latent dimension — the first half is the mean μ, the second half the log-variance log σ². The VAE's sampling layer ([06.2](../README.md)) splits them and draws `z = μ + σ·ε`. So a latent dim of 4 means an 8-wide bottleneck:

In [ ]:
latent_dim = 4
vae_bottleneck = LinearBottleneck(in_features=16, hidden_size=2 * latent_dim)
stats = vae_bottleneck(torch.randn(2, 5, 16))      # (2, 5, 8)
mu, logvar = stats[..., :latent_dim], stats[..., latent_dim:]
print("bottleneck output:", tuple(stats.shape), "= mean + log-variance stacked")
print("  mu     :", tuple(mu.shape))
print("  logvar :", tuple(logvar.shape))
print("the VAE samples z = mu + exp(0.5·logvar)·ε from these — full story in Module 06")

This is why the config's `hidden_sizes` last entry is the *latent* size, but the bottleneck built for a variational composite is `2×` that — a detail the composite builder handles so you rarely compute it by hand.

### 2.4 — The passthrough case

In [ ]:
from neural_data_decoding.models.bottleneck import PassthroughBottleneck

passthrough = PassthroughBottleneck(in_features=16)
x = torch.randn(2, 5, 16)
print("passthrough out:", tuple(passthrough(x).shape), "— unchanged (identity)")
print("equal to input?", torch.equal(passthrough(x), x))

`PassthroughBottleneck` is the "no compression" case — used when the architecture doesn't want a bottleneck (e.g. the Logistic-Regression model, where the classifier reads features directly). Same module *interface* as `LinearBottleneck` (both are `nn.Module`s taking `in_features`), so the composite treats them interchangeably — the [04.1](04.1_architecture_string_dispatcher.ipynb) swap-by-config principle, again.

## Section 3 — The `neural_data_decoding` implementation

`LinearBottleneck` in full — a Linear with explicit He init:

In [ ]:
from pathlib import Path
src = Path("../../src/neural_data_decoding/models/bottleneck.py").read_text().splitlines()
cls = next(j for j, line in enumerate(src) if line.startswith("class LinearBottleneck"))
i = next(j for j in range(cls, len(src)) if src[j].strip().startswith("def __init__"))
for k in range(i, min(i + 16, len(src))):
    print(f"{k + 1:4} | {src[k]}")

Things to spot:

* `nn.init.kaiming_normal_(self.linear.weight, nonlinearity="relu")` — He initialization made *explicit*, matching MATLAB's `"he"` (the default PyTorch init is different — [04.8](04.8_weight_initialization_he_vs_pytorch_defaults.ipynb) is the whole story).
* `nn.init.zeros_(self.linear.bias)` — biases start at zero, also matching MATLAB.
* Input validation in the constructor (`in_features > 0`, `hidden_size > 0`) — fail loudly at build, not at forward.
* No activation in `forward` — just `self.linear(x)`. The deliberate linearity of §2.1.

## Section 4 — Hands-on exercises

### Exercise 4.1 — sequence preservation

Confirm that a `LinearBottleneck(20 → 5)` applied to a `(3, 8, 20)` input preserves the batch and window axes and only changes the last. Then explain why that matters for the LSTM classifier.

In [ ]:
# Your attempt here


In [ ]:
# Reveal:
b = LinearBottleneck(in_features=20, hidden_size=5)
out = b(torch.randn(3, 8, 20))
print("output shape:", tuple(out.shape), "— batch=3 and windows=8 preserved, 20→5 on last axis")
print("The LSTM classifier needs the 8-window SEQUENCE, so the bottleneck must not collapse it.")

### Exercise 4.2 — size the VAE bottleneck

You want a latent dimension of 6 for a stochastic encoder whose encoder outputs 32 hidden units. What `hidden_size` should the `LinearBottleneck` have, and what shape is its output for a `(4, 10, 32)` input? Build it and verify.

In [ ]:
# Reveal:
latent = 6
b = LinearBottleneck(in_features=32, hidden_size=2 * latent)   # 2× for mean+logvar
out = b(torch.randn(4, 10, 32))
print(f"hidden_size = 2 × {latent} = {2*latent}; output {tuple(out.shape)}")
print(f"splits into mu {tuple(out[..., :latent].shape)} + logvar {tuple(out[..., latent:].shape)}")

## Section 5 — Common errors

### The classifier complains the sequence axis is gone

Something collapsed the windows — a `Linear` doesn't (it preserves leading axes, §2.2), but a stray `.mean(dim=1)` or `[:, -1]` would. The bottleneck is per-timestep by design.

### VAE sampling fails: mean/logvar shape mismatch

The bottleneck output width isn't `2 × latent_dim`, so the split into μ and log σ² is uneven. For a variational composite the bottleneck must be `2×` the latent size (§2.3).

### Bottleneck output distribution looks off from the start

Likely the init — PyTorch's default `Linear` init isn't He. `LinearBottleneck` sets He explicitly; if you swap in a bare `nn.Linear`, you lose that ([04.8](04.8_weight_initialization_he_vs_pytorch_defaults.ipynb)).

### `PassthroughBottleneck` changes the shape

It shouldn't — it's an identity on the feature axis. If a shape changed, you built a `LinearBottleneck` by mistake, or the input's last axis wasn't what you expected.

## Section 6 — Further reading

- [nn.Linear docs](https://pytorch.org/docs/stable/generated/torch.nn.Linear.html) — the "applies to last axis, broadcasts over the rest" behavior.
- [Auto-Encoding Variational Bayes](https://arxiv.org/abs/1312.6114) — the VAE paper the `2×latent` bottleneck feeds (deep-dived in Module 06).
- [`src/neural_data_decoding/models/bottleneck.py`](../../src/neural_data_decoding/models/bottleneck.py) — both bottleneck classes.

Next up: **[04.6 the multi-head classifier](04.6_multi_head_classifier.ipynb)** — one output head per target dimension, via `nn.ModuleList`.